# Reddit Mental Health — Multi-Class Topic Classification

**Goal:** classify Reddit mental-health posts into `depression`, `anxiety`,
`crisis`, `neutral`, `loneliness`, and **compare five modeling approaches**
so we can talk concretely about *which model to use and when*:

| # | Model | Family |
|---|-------|--------|
| 1 | TF-IDF + Logistic Regression | Classical ML |
| 2 | RNN (Elman) | Deep learning, sequential |
| 3 | BiLSTM | Deep learning, sequential |
| 4 | DistilBERT (fine-tuned) | Transformer (supervised) |
| 5 | BART-large-mnli (zero-shot) | Transformer (zero-shot) |

Everything — data loading, cleaning, EDA, all five models, and the
comparison — lives in this one notebook by design (presentation flow).
Artifacts each model saves are what `app.py` loads for the live demo.

> **Note on labeling:** Labels (`depression`, `anxiety`, `crisis`, `neutral`,
> `loneliness`) are derived from the source subreddit each post came from —
> not from positive/negative polarity. This is **multi-class topic
> classification**, not sentiment analysis. The zero-shot model (Model 5)
> receives these same label strings as candidate categories with zero
> training examples.


## 1. Setup & Imports

In [ ]:
!pip3 install torch

Defaulting to user installation because normal site-packages is not writeable
ERROR: Invalid requirement: 'torch,'
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
!pip3 install sklearn

Defaulting to user installation because normal site-packages is not writeable
ERROR: Invalid requirement: 'sklearn,'
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [11]:
!pip3 install transformers

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 12.0 MB 7.2 MB/s eta 0:00:01
     |████████████████████████████████| 447 kB 22.6 MB/s eta 0:00:01
     |████████████████████████████████| 566 kB 12.5 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 9.8 MB/s eta 0:00:01
     |████████████████████████████████| 3.9 MB 39.2 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [12]:
import glob
import random
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from wordcloud import WordCloud

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Libraries loaded.")

/Users/sivamanisubrahmanyaharivamsipullipudi/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sivamanisubrahmanyaharivamsipullipudi/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


## 2. Configuration

Everything tunable lives here. `SAMPLES_PER_CLASS` controls how much data
the deep-learning models train on — the cleaned dataset has 130K-520K
rows per class, which would take hours to fine-tune DistilBERT on. We take
a balanced stratified sample instead so **all four models are compared on
the exact same data** and the notebook finishes in well under half an hour.
Raise it later (e.g. for a final report, not a live demo) for stronger
absolute numbers — the relative ranking between models is already clear
at this size.

In [13]:
PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "Original Reddit Data" / "raw data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

for d in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Source subreddit -> label. Built case-insensitively on purpose: the
# original EDA notebook mapped {'anxiety': 'anxiety', ...} against a
# 'subreddit' column that actually contains 'Anxiety' (capital A), so the
# lookup silently missed it (and 'lonely', which wasn't in the map at all)
# and dumped both into a bogus 'other' bucket. Lower-casing both sides
# fixes that.
SUBREDDIT_TO_LABEL = {
    "depression": "depression",
    "anxiety": "anxiety",
    "suicidewatch": "crisis",
    "mentalhealth": "neutral",
    "lonely": "loneliness",
}
LABELS = ["depression", "anxiety", "crisis", "neutral", "loneliness"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

TEXT_COL = "selftext"
LABEL_COL = "label"

SAMPLES_PER_CLASS = 8000   # -> 40,000 total rows shared by every model
                           # Raised from 5000 for stronger numbers while
                           # still finishing well under 45 min on CPU.
MIN_WORD_COUNT = 3

# RNN / LSTM
VOCAB_SIZE = 20_000
MAX_LEN_WORDS = 120
EMBED_DIM = 128
HIDDEN_DIM = 128
RNN_EPOCHS = 5
RNN_BATCH_SIZE = 64
RNN_LR = 1e-3

# DistilBERT (fine-tuned)
BERT_MODEL_NAME = "distilbert-base-uncased"
BERT_MAX_LEN = 128
BERT_EPOCHS = 2
BERT_BATCH_SIZE = 16
BERT_LR = 2e-5

# Zero-shot (no training -- NLI entailment over candidate label strings)
ZEROSHOT_MODEL_NAME = "facebook/bart-large-mnli"
ZEROSHOT_EVAL_SAMPLES = 500   # subset for speed on CPU; raise for final report

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {DEVICE}")

results_log = []  # each entry: {model, split, accuracy, macro_f1, train_time_s, infer_ms_per_sample}


Device: mps


## 3-5. Load, Clean & Label — Streamed (Memory-Bounded)

The raw data is ~1.85M rows across 219 files. Loading and cleaning that
all at once as one giant DataFrame comfortably blew past 16GB of RAM on a
laptop (it's what crashed the machine on an earlier attempt at this
notebook). Since we only need a balanced `SAMPLES_PER_CLASS`-per-class
sample for training anyway, we instead **stream one file at a time**:
clean + label + filter each file's rows immediately, keep a running
per-class pool capped at `POOL_CAP_PER_CLASS`, and never hold the full
1.85M-row corpus in memory at once. Peak memory stays bounded regardless
of how many raw files there are.

This cell also does the "before cleaning" EDA bookkeeping (subreddit
counts, drop reasons) as lightweight running counters instead of on a
giant DataFrame.

In [14]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MD_LINK_RE = re.compile(r"\[([^\]]*)\]\([^)]*\)")
SUB_MENTION_RE = re.compile(r"/?r/\w+")
USER_MENTION_RE = re.compile(r"/?u/\w+")
NON_PRINTABLE_RE = re.compile(r"[^\x20-\x7E\n]")
WHITESPACE_RE = re.compile(r"\s+")
JUNK_VALUES = {"[deleted]", "[removed]", "", "nan", "none"}


def is_junk(text) -> bool:
    return text is None or str(text).strip().lower() in JUNK_VALUES


def clean_text(text) -> str:
    if text is None:
        return ""
    text = str(text)
    text = MD_LINK_RE.sub(r"\1", text)
    text = URL_RE.sub(" ", text)
    text = SUB_MENTION_RE.sub(" ", text)
    text = USER_MENTION_RE.sub(" ", text)
    text = NON_PRINTABLE_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text)
    return text.strip()


files = sorted(glob.glob(str(RAW_DATA_DIR / "**" / "*.csv"), recursive=True))
print(f"Found {len(files)} raw CSV files")

lookup = {k.lower(): v for k, v in SUBREDDIT_TO_LABEL.items()}
POOL_CAP_PER_CLASS = SAMPLES_PER_CLASS * 6  # generous working pool, bounds memory well below full corpus

raw_subreddit_counts = Counter()
n_raw_rows = 0
n_dropped_junk = 0
n_dropped_unrecognized = 0
n_dropped_short = 0
class_pools = {label: [] for label in LABELS}
rng = np.random.default_rng(RANDOM_SEED)

for fi, f in enumerate(files):
    try:
        chunk = pd.read_csv(f, usecols=[TEXT_COL, "subreddit"], on_bad_lines="skip", low_memory=False)
    except ValueError:
        continue

    n_raw_rows += len(chunk)
    raw_subreddit_counts.update(chunk["subreddit"].astype(str).str.strip().str.lower().value_counts().to_dict())

    junk_mask = chunk[TEXT_COL].apply(is_junk)
    n_dropped_junk += int(junk_mask.sum())
    chunk = chunk.loc[~junk_mask].copy()

    normalized_sub = chunk["subreddit"].astype(str).str.strip().str.lower()
    chunk[LABEL_COL] = normalized_sub.map(lookup)
    unrecognized_mask = chunk[LABEL_COL].isna()
    n_dropped_unrecognized += int(unrecognized_mask.sum())
    chunk = chunk.loc[~unrecognized_mask].copy()

    chunk[TEXT_COL] = chunk[TEXT_COL].apply(clean_text)
    chunk["word_count"] = chunk[TEXT_COL].str.split().apply(len)
    short_mask = chunk["word_count"] < MIN_WORD_COUNT
    n_dropped_short += int(short_mask.sum())
    chunk = chunk.loc[~short_mask, [TEXT_COL, LABEL_COL, "word_count"]]

    for label, group in chunk.groupby(LABEL_COL):
        class_pools[label].append(group)

    # periodically trim each class pool back down so memory stays bounded
    if fi % 15 == 0 or fi == len(files) - 1:
        for label in LABELS:
            if not class_pools[label]:
                continue
            pooled = pd.concat(class_pools[label], ignore_index=True).drop_duplicates(subset=[TEXT_COL])
            if len(pooled) > POOL_CAP_PER_CLASS:
                pooled = pooled.sample(n=POOL_CAP_PER_CLASS, random_state=RANDOM_SEED)
            class_pools[label] = [pooled]

print(f"\nProcessed {n_raw_rows:,} raw rows")
print(f"Dropped {n_dropped_junk:,} deleted/removed/empty posts")
print(f"Dropped {n_dropped_unrecognized:,} rows with unrecognized/garbled subreddit values")
print(f"Dropped {n_dropped_short:,} posts shorter than {MIN_WORD_COUNT} words")

df = pd.concat([class_pools[label][0] for label in LABELS if class_pools[label]], ignore_index=True)
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
del class_pools
print(f"\nFinal pooled dataset (capped at {POOL_CAP_PER_CLASS:,}/class): {len(df):,} rows")
print(df[LABEL_COL].value_counts())

Found 0 raw CSV files

Processed 0 raw rows
Dropped 0 deleted/removed/empty posts
Dropped 0 rows with unrecognized/garbled subreddit values
Dropped 0 posts shorter than 3 words


ValueError: No objects to concatenate

### EDA on what we just loaded

Raw subreddit counts (before cleaning) show the same long tail of garbled
one-off values discussed above — real signal that the fix for the
case-sensitive label bug plus dropping unrecognized subreddits was
needed.

In [ ]:
top_counts = pd.Series(raw_subreddit_counts).sort_values(ascending=False).head(6)
plt.figure(figsize=(7, 4))
sns.barplot(x=top_counts.index, y=top_counts.values, hue=top_counts.index, palette="Blues_d", legend=False)
plt.title("Raw Post Counts — Top Subreddit Values (lower-cased)")
plt.ylabel("count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print(f"Distinct subreddit values seen: {len(raw_subreddit_counts):,} (only {len(LABELS)} are real)")

In [ ]:
plt.figure(figsize=(8, 4))
df["word_count"].clip(upper=500).hist(bins=60, color="steelblue")
plt.title("Cleaned Post Word-Count Distribution (clipped at 500)")
plt.xlabel("words per post")
plt.tight_layout()
plt.show()
print(df["word_count"].describe())

In [ ]:
counts = df[LABEL_COL].value_counts().reindex(LABELS)
plt.figure(figsize=(7, 4))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False)
plt.title("Cleaned Label Distribution (pooled sample)")
plt.ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
stop_words = set(stopwords.words("english"))
fig, axes = plt.subplots(1, len(LABELS), figsize=(4 * len(LABELS), 4))
for ax, label in zip(axes, LABELS):
    sample_text = " ".join(df.loc[df[LABEL_COL] == label, TEXT_COL].sample(
        min(3000, (df[LABEL_COL] == label).sum()), random_state=RANDOM_SEED
    ))
    wc = WordCloud(width=300, height=250, stopwords=stop_words, background_color="white", max_words=60).generate(sample_text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(label)
    ax.axis("off")
plt.suptitle("Most Common Words per Class")
plt.tight_layout()
plt.show()

## 6. Stratified Sample + Train/Val/Test Split

We sample `SAMPLES_PER_CLASS` posts per class (balanced, so no class gets
an unfair advantage), then split 70/15/15. **All four models below train
and get evaluated on this exact same split** — that's what makes the
final comparison fair.

In [ ]:
sampled = (
    df.groupby(LABEL_COL, group_keys=False)
    .apply(lambda g: g.sample(min(len(g), SAMPLES_PER_CLASS), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)
print("Sampled distribution:\n", sampled[LABEL_COL].value_counts())

train_df, temp_df = train_test_split(sampled, test_size=0.30, stratify=sampled[LABEL_COL], random_state=RANDOM_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df[LABEL_COL], random_state=RANDOM_SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\ntrain={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

train_df[[TEXT_COL, LABEL_COL]].to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df[[TEXT_COL, LABEL_COL]].to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df[[TEXT_COL, LABEL_COL]].to_csv(PROCESSED_DIR / "test.csv", index=False)

## 7. Shared Evaluation Helper

Used identically by every model so the comparison in Section 12 is
apples-to-apples.

In [ ]:
def evaluate_predictions(y_true, y_pred, model_name, split, train_time_s=None, infer_ms=None, plot=True):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average="macro")

    print(f"\n=== {model_name} — {split} ===")
    print(classification_report(y_true, y_pred, labels=LABELS, digits=3))

    if plot:
        cm = confusion_matrix(y_true, y_pred, labels=LABELS)
        plt.figure(figsize=(5.5, 4.5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"{model_name} — {split} confusion matrix")
        plt.tight_layout()
        fig_path = REPORTS_DIR / f"{model_name.lower().replace(' ', '_')}_{split}_confusion_matrix.png"
        plt.savefig(fig_path, dpi=130)
        plt.show()

    results_log.append({
        "model": model_name,
        "split": split,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "train_time_s": train_time_s,
        "infer_ms_per_sample": infer_ms,
    })
    return acc, macro_f1


@torch.no_grad()
def measure_inference_latency_torch(predict_one_fn, sample_texts, n=200):
    sample_texts = list(sample_texts)[:n]
    t0 = time.time()
    for t in sample_texts:
        predict_one_fn(t)
    return (time.time() - t0) / len(sample_texts) * 1000  # ms/sample

## 8. Model 1 — TF-IDF + Logistic Regression (classical ML baseline)

Fast, interpretable, needs no GPU. `SGDClassifier(loss="log_loss")` is
logistic regression trained by SGD — scales far better than plain
`LogisticRegression` and, unlike `LinearSVC`, still gives us
`predict_proba` for the confidence bars in `app.py`.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True, stop_words="english")

t0 = time.time()
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df[TEXT_COL])
X_val_tfidf = tfidf_vectorizer.transform(val_df[TEXT_COL])
X_test_tfidf = tfidf_vectorizer.transform(test_df[TEXT_COL])

# alpha raised 1e-6 -> 1e-4 (better regularization at moderate sample sizes);
# max_iter raised 50 -> 100 to ensure full convergence.
tfidf_clf = SGDClassifier(loss="log_loss", class_weight="balanced", alpha=1e-4, max_iter=100, random_state=RANDOM_SEED)
tfidf_clf.fit(X_train_tfidf, train_df[LABEL_COL])
tfidf_train_time = time.time() - t0
print(f"Vocab size: {len(tfidf_vectorizer.vocabulary_):,}   trained in {tfidf_train_time:.1f}s")

val_preds = tfidf_clf.predict(X_val_tfidf)
evaluate_predictions(val_df[LABEL_COL], val_preds, "TF-IDF", "val", train_time_s=tfidf_train_time)

test_preds = tfidf_clf.predict(X_test_tfidf)


def tfidf_predict_one(text):
    return tfidf_clf.predict(tfidf_vectorizer.transform([text]))[0]


infer_ms = measure_inference_latency_torch(tfidf_predict_one, test_df[TEXT_COL])
evaluate_predictions(test_df[LABEL_COL], test_preds, "TF-IDF", "test", infer_ms=infer_ms)

BASELINE_DIR = MODELS_DIR / "baseline"
BASELINE_DIR.mkdir(exist_ok=True)
joblib.dump(tfidf_vectorizer, BASELINE_DIR / "tfidf_vectorizer.joblib")
joblib.dump(tfidf_clf, BASELINE_DIR / "sgd_classifier.joblib")
print("Saved TF-IDF vectorizer + classifier ->", BASELINE_DIR)


## 9. Shared Vocabulary for RNN & LSTM

Both sequence models tokenize with the same simple whitespace/word
tokenizer and share one vocabulary (top `VOCAB_SIZE` most frequent
training-set words), so they're an apples-to-apples comparison of
*architecture* (plain RNN vs LSTM), not of tokenization.

In [ ]:
TOKEN_RE = re.compile(r"[a-z']+")


def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())


counter = Counter()
for text in train_df[TEXT_COL]:
    counter.update(tokenize(text))

# reserve 0=<pad>, 1=<unk>
vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab):,} (from {len(counter):,} unique training words)")


def encode(text: str, max_len: int = MAX_LEN_WORDS):
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    length = max(len(ids), 1)
    ids = ids + [vocab["<pad>"]] * (max_len - len(ids))
    return ids, length


class SequenceDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN_WORDS):
        self.encoded = [encode(t, max_len) for t in texts]
        self.labels = [LABEL2ID[l] for l in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids, length = self.encoded[idx]
        return torch.tensor(ids, dtype=torch.long), length, self.labels[idx]


def seq_collate(batch):
    ids, lengths, labels = zip(*batch)
    return torch.stack(ids), torch.tensor(lengths, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


train_seq_ds = SequenceDataset(train_df[TEXT_COL], train_df[LABEL_COL])
val_seq_ds = SequenceDataset(val_df[TEXT_COL], val_df[LABEL_COL])
test_seq_ds = SequenceDataset(test_df[TEXT_COL], test_df[LABEL_COL])

train_seq_loader = DataLoader(train_seq_ds, batch_size=RNN_BATCH_SIZE, shuffle=True, collate_fn=seq_collate)
val_seq_loader = DataLoader(val_seq_ds, batch_size=RNN_BATCH_SIZE * 2, shuffle=False, collate_fn=seq_collate)
test_seq_loader = DataLoader(test_seq_ds, batch_size=RNN_BATCH_SIZE * 2, shuffle=False, collate_fn=seq_collate)

class_counts = train_df[LABEL_COL].value_counts()
seq_class_weights = torch.tensor(
    [len(train_df) / (len(LABELS) * class_counts.get(l, 1)) for l in LABELS], dtype=torch.float
).to(DEVICE)
print("Class weights (for imbalance):", dict(zip(LABELS, seq_class_weights.tolist())))

## 10. Model 2 — Simple RNN (PyTorch)

A single-layer Elman RNN. This is the weakest of the sequence models on
purpose — it's included as a baseline "deep learning" model so the
presentation can show *why* LSTM/attention-based models were invented
(vanishing gradients on longer posts, weaker long-range dependency
handling).

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        return self.fc(self.dropout(hidden[-1]))


def train_sequence_model(model, train_loader, val_loader, epochs, lr, class_weights, model_label):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    epoch_losses = []   # training loss per epoch -- for loss curve
    val_accs = []       # val accuracy per epoch -- for accuracy curve

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for ids, lengths, labels in train_loader:
            ids, labels = ids.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(ids, lengths)
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_preds, val_labels = [], []
            for ids, lengths, labels in val_loader:
                ids = ids.to(DEVICE)
                logits = model(ids, lengths)
                val_preds.extend(logits.argmax(dim=-1).cpu().tolist())
                val_labels.extend(labels.tolist())
        val_acc = accuracy_score(val_labels, val_preds)
        avg_loss = running_loss / len(train_loader)
        epoch_losses.append(avg_loss)
        val_accs.append(val_acc)
        print(f"  [{model_label}] epoch {epoch}/{epochs}  train_loss={avg_loss:.4f}  val_acc={val_acc:.4f}")

    # Loss & accuracy curves
    epoch_range = range(1, epochs + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax1.plot(epoch_range, epoch_losses, marker="o", color="steelblue")
    ax1.set_title(f"{model_label} -- Training Loss per Epoch")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-Entropy Loss")
    ax2.plot(epoch_range, val_accs, marker="o", color="darkorange")
    ax2.set_title(f"{model_label} -- Val Accuracy per Epoch")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / f"{model_label.lower()}_training_curves.png", dpi=130)
    plt.show()

    return time.time() - t0


@torch.no_grad()
def predict_sequence_model(model, loader):
    model.eval()
    preds, labels_out = [], []
    for ids, lengths, labels in loader:
        ids = ids.to(DEVICE)
        logits = model(ids, lengths)
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
        labels_out.extend(labels.tolist())
    return [ID2LABEL[p] for p in preds], [ID2LABEL[l] for l in labels_out]


rnn_model = RNNClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS))
rnn_train_time = train_sequence_model(rnn_model, train_seq_loader, val_seq_loader, RNN_EPOCHS, RNN_LR, seq_class_weights, "RNN")

val_labels, val_preds = predict_sequence_model(rnn_model, val_seq_loader)
evaluate_predictions(val_labels, val_preds, "RNN", "val", train_time_s=rnn_train_time)

test_labels, test_preds = predict_sequence_model(rnn_model, test_seq_loader)


def rnn_predict_one(text):
    ids, length = encode(text)
    ids_t = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = rnn_model(ids_t, torch.tensor([length]))
    return ID2LABEL[int(logits.argmax(dim=-1).item())]


infer_ms = measure_inference_latency_torch(rnn_predict_one, test_df[TEXT_COL])
evaluate_predictions(test_labels, test_preds, "RNN", "test", infer_ms=infer_ms)

RNN_DIR = MODELS_DIR / "rnn"
RNN_DIR.mkdir(exist_ok=True)
torch.save(rnn_model.state_dict(), RNN_DIR / "rnn_state_dict.pt")
joblib.dump(vocab, RNN_DIR / "vocab.joblib")
print("Saved RNN model + vocab ->", RNN_DIR)


## 11. Model 3 — BiLSTM (PyTorch)

Same vocabulary and training loop shape as the RNN, swapped to a
bidirectional LSTM: gated memory cells handle longer posts and
longer-range dependencies much better than a plain RNN.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        # hidden: (num_directions, batch, hidden_dim) for a single-layer LSTM
        combined = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(combined))


lstm_model = LSTMClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS))
lstm_train_time = train_sequence_model(lstm_model, train_seq_loader, val_seq_loader, RNN_EPOCHS, RNN_LR, seq_class_weights, "LSTM")

val_labels, val_preds = predict_sequence_model(lstm_model, val_seq_loader)
evaluate_predictions(val_labels, val_preds, "LSTM", "val", train_time_s=lstm_train_time)

test_labels, test_preds = predict_sequence_model(lstm_model, test_seq_loader)


def lstm_predict_one(text):
    ids, length = encode(text)
    ids_t = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = lstm_model(ids_t, torch.tensor([length]))
    return ID2LABEL[int(logits.argmax(dim=-1).item())]


infer_ms = measure_inference_latency_torch(lstm_predict_one, test_df[TEXT_COL])
evaluate_predictions(test_labels, test_preds, "LSTM", "test", infer_ms=infer_ms)

LSTM_DIR = MODELS_DIR / "lstm"
LSTM_DIR.mkdir(exist_ok=True)
torch.save(lstm_model.state_dict(), LSTM_DIR / "lstm_state_dict.pt")
joblib.dump(vocab, LSTM_DIR / "vocab.joblib")
print("Saved LSTM model + vocab ->", LSTM_DIR)

## 12. Model 4 — DistilBERT (fine-tuned transformer)

Pretrained subword tokenizer + transformer encoder, fine-tuned on our
labels. This is the model most likely to win on accuracy since it brings
real semantic/contextual understanding rather than bag-of-words or a
small trained-from-scratch embedding — at the cost of size, training
time, and inference latency.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
).to(DEVICE)


class TextLabelDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = [LABEL2ID[l] for l in labels]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def bert_collate(batch):
    texts, labels = zip(*batch)
    encoded = bert_tokenizer(list(texts), padding=True, truncation=True, max_length=BERT_MAX_LEN, return_tensors="pt")
    encoded["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoded


bert_train_loader = DataLoader(TextLabelDataset(train_df[TEXT_COL], train_df[LABEL_COL]), batch_size=BERT_BATCH_SIZE, shuffle=True, collate_fn=bert_collate)
bert_val_loader = DataLoader(TextLabelDataset(val_df[TEXT_COL], val_df[LABEL_COL]), batch_size=BERT_BATCH_SIZE * 2, shuffle=False, collate_fn=bert_collate)
bert_test_loader = DataLoader(TextLabelDataset(test_df[TEXT_COL], test_df[LABEL_COL]), batch_size=BERT_BATCH_SIZE * 2, shuffle=False, collate_fn=bert_collate)

bert_loss_fn = nn.CrossEntropyLoss(weight=seq_class_weights)
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps = len(bert_train_loader) * BERT_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)


@torch.no_grad()
def bert_eval(loader):
    bert_model.eval()
    preds, labels_out = [], []
    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = bert_model(**batch).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
        labels_out.extend(labels.tolist())
    return [ID2LABEL[p] for p in preds], [ID2LABEL[l] for l in labels_out]


bert_epoch_losses = []   # training loss per epoch
bert_val_accs = []       # val accuracy per epoch

t0 = time.time()
for epoch in range(1, BERT_EPOCHS + 1):
    bert_model.train()
    running_loss = 0.0
    for step, batch in enumerate(bert_train_loader, start=1):
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = bert_model(**batch).logits
        loss = bert_loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch} step {step}/{len(bert_train_loader)}  avg_loss={running_loss / step:.4f}")

    val_labels, val_preds = bert_eval(bert_val_loader)
    val_acc = accuracy_score(val_labels, val_preds)
    avg_loss = running_loss / len(bert_train_loader)
    bert_epoch_losses.append(avg_loss)
    bert_val_accs.append(val_acc)
    print(f"[BERT] epoch {epoch}/{BERT_EPOCHS}  train_loss={avg_loss:.4f}  val_acc={val_acc:.4f}")

bert_train_time = time.time() - t0

# DistilBERT training curves
epoch_range = range(1, BERT_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(epoch_range, bert_epoch_losses, marker="o", color="steelblue")
ax1.set_title("DistilBERT -- Training Loss per Epoch")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-Entropy Loss")
ax2.plot(epoch_range, bert_val_accs, marker="o", color="green")
ax2.set_title("DistilBERT -- Val Accuracy per Epoch")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "distilbert_training_curves.png", dpi=130)
plt.show()


In [ ]:
val_labels, val_preds = bert_eval(bert_val_loader)
evaluate_predictions(val_labels, val_preds, "DistilBERT", "val", train_time_s=bert_train_time)

test_labels, test_preds = bert_eval(bert_test_loader)


def bert_predict_one(text):
    inputs = bert_tokenizer(text, truncation=True, max_length=BERT_MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    return ID2LABEL[int(logits.argmax(dim=-1).item())]


infer_ms = measure_inference_latency_torch(bert_predict_one, test_df[TEXT_COL], n=100)
evaluate_predictions(test_labels, test_preds, "DistilBERT", "test", infer_ms=infer_ms)

BERT_DIR = MODELS_DIR / "distilbert"
BERT_DIR.mkdir(exist_ok=True)
bert_model.save_pretrained(BERT_DIR)
bert_tokenizer.save_pretrained(BERT_DIR)
print("Saved DistilBERT model + tokenizer ->", BERT_DIR)

## 13. Model 5 — Zero-Shot Classification (BART-large-mnli)

No training. No labeled examples. This is the **zero-shot** approach:
a pretrained NLI (Natural Language Inference) model decides whether a
post *entails* each candidate label, then picks the highest-scoring one.

This directly addresses **custom categories, no retraining** — you can
change the label list in `LABELS` and re-run this cell without touching
any training data.

**Why this matters for the demo:** DistilBERT (Model 4) needed
`SAMPLES_PER_CLASS x 5` labeled Reddit posts to learn these categories.
Zero-Shot needs exactly zero. The accuracy gap between them is the cost
of having no training data — and the gap is often surprisingly small on
well-named, semantically distinct labels like these.

> **Speed note:** `ZEROSHOT_EVAL_SAMPLES` (set in Section 2 config)
> limits how many test rows we classify here. Full inference over 6,000
> rows on CPU takes 20-40 minutes. Keep it at 500 for a live demo;
> raise it to `len(test_df)` for a final report.


In [ ]:
from transformers import pipeline

print(f"Loading zero-shot classifier: {ZEROSHOT_MODEL_NAME}")
print("No fine-tuning -- pure pretrained NLI entailment inference")
zs_pipeline = pipeline(
    "zero-shot-classification",
    model=ZEROSHOT_MODEL_NAME,
    device=-1,   # CPU; change to 0 if GPU is available
)

# Use a subset of the test set for speed (controlled by ZEROSHOT_EVAL_SAMPLES)
zs_test_texts  = test_df[TEXT_COL].head(ZEROSHOT_EVAL_SAMPLES).tolist()
zs_test_labels = test_df[LABEL_COL].head(ZEROSHOT_EVAL_SAMPLES).tolist()

print(f"\nRunning zero-shot inference on {len(zs_test_texts):,} test samples...")
t0 = time.time()
zs_preds = []
for i, text in enumerate(zs_test_texts):
    result = zs_pipeline(text[:512], candidate_labels=LABELS)
    zs_preds.append(result["labels"][0])   # highest-scoring label
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(zs_test_texts)} done  ({elapsed:.0f}s elapsed)")

zs_infer_ms = (time.time() - t0) / len(zs_test_texts) * 1000

evaluate_predictions(
    zs_test_labels, zs_preds,
    "Zero-Shot (BART)", "test",
    train_time_s=0,
    infer_ms=zs_infer_ms,
)

print("\n" + "="*60)
print("KEY TAKEAWAY FOR DEMO")
print("="*60)
print(f"Zero-Shot  -- training examples used: 0")
print(f"DistilBERT -- training examples used: {len(train_df):,}")
print("Accuracy gap above = the cost of zero labeled data.")
print("If that gap is acceptable, zero-shot eliminates all data")
print("collection, annotation, and retraining entirely.")


## 14. Model Comparison

Same train/val/test split, same evaluation function, for all five models
-- so this table is a fair head-to-head.


In [ ]:
results_df = pd.DataFrame(results_log)
test_results = results_df[results_df["split"] == "test"].copy()
train_results = results_df[results_df["split"] == "val"][["model", "train_time_s"]].rename(columns={"train_time_s": "train_time_s_"})
test_results = test_results.merge(train_results, on="model", how="left")
test_results["train_time_s"] = test_results["train_time_s"].fillna(test_results["train_time_s_"])
test_results = test_results.drop(columns=["train_time_s_", "split"])
test_results = test_results.sort_values("macro_f1", ascending=False).reset_index(drop=True)
test_results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.barplot(data=test_results, x="model", y="accuracy", hue="model", palette="crest", ax=axes[0], legend=False)
axes[0].set_title("Test Accuracy")
axes[0].set_ylim(0, 1)

sns.barplot(data=test_results, x="model", y="macro_f1", hue="model", palette="crest", ax=axes[1], legend=False)
axes[1].set_title("Test Macro F1")
axes[1].set_ylim(0, 1)

sns.barplot(data=test_results, x="model", y="infer_ms_per_sample", hue="model", palette="flare", ax=axes[2], legend=False)
axes[2].set_title("Inference latency (ms/sample, CPU/MPS single-item)")

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "model_comparison.png", dpi=130)
plt.show()

### Reading the comparison -- which model to use, when

- **TF-IDF + Logistic Regression** -- strongest accuracy-per-second here.
  No GPU, trains in seconds, sub-millisecond inference, and it's directly
  interpretable (you can inspect top weighted n-grams per class). Use it
  when you need a fast, cheap, explainable baseline, for on-device /
  low-resource inference, or whenever the marginal accuracy of a deep
  model isn't worth the extra latency and infrastructure.
- **Simple RNN** -- usually the weakest deep model on this kind of text.
  Included mainly to demonstrate *why* LSTM/transformers exist: a plain
  RNN struggles to carry signal across long posts (vanishing gradients).
  Rarely the right production choice today.
- **BiLSTM** -- a real step up from the RNN on longer, more nuanced posts,
  still lightweight enough to train from scratch quickly. Reasonable
  middle ground when you want more sequence-awareness than TF-IDF but
  can't afford transformer-scale training/serving cost.
- **DistilBERT (fine-tuned)** -- should post the best accuracy/macro-F1
  (pretrained language understanding beats a small from-scratch embedding),
  at the cost of being the slowest to train and slowest per inference.
  Use it when accuracy matters most and you have GPU budget.
- **Zero-Shot (BART-large-mnli)** -- no training data at all. Uses NLI
  entailment to score each candidate label against the input text, then
  picks the top-scoring one. Lower accuracy than fine-tuned DistilBERT,
  but works on any custom label list instantly -- this is the
  'custom categories, no retraining' approach. If the accuracy gap vs.
  DistilBERT is acceptable, zero-shot eliminates all data collection
  and annotation cost entirely.

*(Exact numbers above come from the SAMPLES_PER_CLASS sample configured
in Section 2 -- rerun with a larger sample for a final report; the
ranking is expected to hold.)*


## 15. Artifacts Saved for app.py

- models/baseline/tfidf_vectorizer.joblib, models/baseline/sgd_classifier.joblib
- models/rnn/rnn_state_dict.pt, models/rnn/vocab.joblib
- models/lstm/lstm_state_dict.pt, models/lstm/vocab.joblib
- models/distilbert/ (HF save_pretrained directory)
- reports/ -- confusion matrices, training curves, and comparison chart
- data/processed/{train,val,test}.csv -- the shared split used above

app.py loads all four supervised models and shows every model's prediction
side by side. The zero-shot model (Model 5) loads directly from HuggingFace
at runtime -- no saved artifact needed.
